# 04 群聚調查工作流 — 參考解答

松柏護理之家退伍軍人症群聚 SitRep 練習的完整解答。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150


## 題目 1：摘要指標

In [ ]:
df = pd.read_csv("data/synthetic/legionella_outbreak.csv")

# 日期轉換
date_cols = [
    "facility_admission_date", "symptom_onset_date",
    "hospitalization_date", "death_date", "notification_date",
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

total = len(df)
infected = df["infected"].sum()
confirmed = (df["case_classification"] == "confirmed").sum()
probable = (df["case_classification"] == "probable").sum()
hospitalized = df["hospitalized"].sum()
icu = df["icu_admission"].sum()
deaths = (df["outcome"] == "dead").sum()

print("=" * 50)
print("松柏護理之家退伍軍人症群聚 — SitRep")
print("=" * 50)
print(f"住民總數：{total}")
print(f"感染人數：{infected}（侵襲率 {infected/total:.1%}）")
print(f"  確診：{confirmed}　可能：{probable}")
print(f"住院：{hospitalized}（住院率 {hospitalized/infected:.1%}）")
print(f"ICU：{icu}（ICU 率 {icu/hospitalized:.1%}）")
print(f"死亡：{deaths}（CFR {deaths/infected:.1%}）")

## 題目 2：人時地三要素

In [ ]:
# --- 人 (Person) ---
cases = df[df["infected"] == 1]

print("=== 人口學特徵（感染者）===")
print(f"年齡中位數：{cases['age'].median():.0f} 歲"
      f"（範圍 {cases['age'].min()}-{cases['age'].max()}）")
print(f"男性比例：{(cases['sex'] == 'M').mean():.1%}")

In [ ]:
import matplotlib.dates as mdates

# --- 時 (Time) ---
daily = cases.groupby("symptom_onset_date").size().rename("cases")

# 加入爆發前背景期（含零病例日）
date_range = pd.date_range(
    daily.index.min() - pd.Timedelta(days=3),
    daily.index.max() + pd.Timedelta(days=1),
)
daily = daily.reindex(date_range, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(
    daily.index, daily.values,
    width=1.0,
    color="#2c7fb8", edgecolor="white", linewidth=0.5,
)
ax.set_title(
    "松柏護理之家退伍軍人症流行曲線，依發病日，2026 年 1 月",
    fontsize=13, fontweight="bold",
)
ax.set_xlabel("發病日期（Date of Symptom Onset）")
ax.set_ylabel("病例數（Number of Cases）")

ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
fig.autofmt_xdate(rotation=45, ha="right")

ax.set_ylim(bottom=0)
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

# 使用原始（不含背景期）的 daily 序列算高峰
daily_cases = cases.groupby("symptom_onset_date").size().rename("cases")
print(f"流行期間：{daily_cases.index.min().date()} – {daily_cases.index.max().date()}")
print(f"高峰日：{daily_cases.idxmax().date()}（{daily_cases.max()} 例）")

In [ ]:
# --- 地 (Place) ---
wing_stats = (
    df.groupby(["floor", "wing"])
    .agg(
        residents=("case_id", "size"),
        infected=("infected", "sum"),
        deaths=("outcome", lambda x: (x == "dead").sum()),
    )
    .reset_index()
)
wing_stats["AR%"] = (wing_stats["infected"] / wing_stats["residents"] * 100).round(1)
wing_stats["CFR%"] = (wing_stats["deaths"] / wing_stats["infected"] * 100).round(1)
wing_stats["label"] = wing_stats["floor"].astype(str) + wing_stats["wing"]

print("=== 各翼區疫情摘要 ===")
print(wing_stats[["label", "residents", "infected", "AR%", "deaths", "CFR%"]]
      .to_string(index=False))

## 題目 3：按年齡組的分層摘要

In [ ]:
df["age_group"] = pd.cut(
    df["age"], bins=[59, 69, 79, 89, 100],
    labels=["60-69", "70-79", "80-89", "90+"],
)

age_stats = (
    df.groupby("age_group", observed=True)
    .agg(
        residents=("case_id", "size"),
        infected=("infected", "sum"),
        deaths=("outcome", lambda x: (x == "dead").sum()),
    )
)
age_stats["AR%"] = (age_stats["infected"] / age_stats["residents"] * 100).round(1)
age_stats["CFR%"] = (age_stats["deaths"] / age_stats["infected"] * 100).round(1)

print("=== 年齡組分層摘要 ===")
print(age_stats.to_string())

print(f"\n侵襲率最高：{age_stats['AR%'].idxmax()}（{age_stats['AR%'].max()}%）")
print(f"CFR 最高：{age_stats['CFR%'].idxmax()}（{age_stats['CFR%'].max()}%）")
print("→ 侵襲率最高的年齡組不一定 CFR 最高——")
print("  侵襲率反映『感染風險』，CFR 反映『感染後的預後』，兩者受不同因子影響。")

## 題目 4（挑戰題）：generate_sitrep 函式

In [ ]:
def generate_sitrep(csv_path):
    """從 CSV 產出 SitRep 摘要字典。"""
    df = pd.read_csv(csv_path)
    for col in ["symptom_onset_date", "hospitalization_date",
                "death_date", "notification_date"]:
        df[col] = pd.to_datetime(df[col], errors="coerce")
    df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

    total = len(df)
    infected_n = int(df["infected"].sum())
    deaths_n = int((df["outcome"] == "dead").sum())

    # 高峰日
    cases = df[df["infected"] == 1]
    daily = cases.groupby("symptom_onset_date").size()
    peak_date = str(daily.idxmax().date()) if len(daily) > 0 else None

    # 侵襲率最高翼區
    ws = (
        df.groupby(["floor", "wing"])
        .agg(residents=("case_id", "size"), infected=("infected", "sum"))
        .reset_index()
    )
    ws["ar"] = ws["infected"] / ws["residents"]
    worst = ws.loc[ws["ar"].idxmax()]
    worst_wing = f"{worst['floor']}{worst['wing']}"

    return {
        "total_residents": total,
        "infected": infected_n,
        "attack_rate": round(infected_n / total * 100, 1),
        "deaths": deaths_n,
        "cfr": round(deaths_n / infected_n * 100, 1) if infected_n else 0,
        "hospitalized": int(df["hospitalized"].sum()),
        "icu": int(df["icu_admission"].sum()),
        "peak_date": peak_date,
        "worst_wing": worst_wing,
    }

result = generate_sitrep("data/synthetic/legionella_outbreak.csv")
print("=== SitRep 結構化輸出 ===")
for k, v in result.items():
    print(f"  {k}: {v}")

## 題目 5：產出一份 Word 報告

In [ ]:
from io import BytesIO
from datetime import datetime
from docx import Document
from docx.shared import Inches

pathlib.Path("output").mkdir(exist_ok=True)

# 將流行曲線存入記憶體
epicurve_buf = BytesIO()
fig.savefig(epicurve_buf, format="png", dpi=150, bbox_inches="tight")
epicurve_buf.seek(0)

report_time = datetime.now().strftime("%Y-%m-%d %H:%M")

doc = Document()
doc.add_heading("松柏護理之家退伍軍人症 SitRep", level=1)
doc.add_paragraph(f"報告時間：{report_time}")

# 摘要指標表格
doc.add_heading("摘要指標", level=2)
table = doc.add_table(rows=5, cols=2, style="Light Grid Accent 1")
for i, (label, value) in enumerate([
    ("住民總數", str(total)),
    ("感染人數", f"{infected}（侵襲率 {infected/total:.1%}）"),
    ("確診 / 可能", f"{confirmed} / {probable}"),
    ("住院 / ICU", f"{hospitalized} / {icu}"),
    ("死亡", f"{deaths}（CFR {deaths/infected:.1%}）"),
]):
    table.rows[i].cells[0].text = label
    table.rows[i].cells[1].text = value

# 嵌入流行曲線
doc.add_heading("流行曲線", level=2)
epicurve_buf.seek(0)
doc.add_picture(epicurve_buf, width=Inches(6))

doc.save("output/my_sitrep.docx")
print("Word 報告已儲存：output/my_sitrep.docx")

### 解讀

- **侵襲率 ~43%**：非常高，代表疫情嚴重且暴露源廣泛
- **CFR ~16%**：退伍軍人症在護理之家族群的 CFR 偏高，與文獻一致
- **3B 翼侵襲率最高**：需要優先調查該翼區的供水系統和淋浴設備
- **年齡組差異**：侵襲率最高和 CFR 最高可能不在同一年齡組，這表示「感染風險」和「預後」受不同因子影響

## 題目 6 解答

In [ ]:
import numpy as np
from epi_learning.metrics import attack_rate, risk_ratio
from epi_learning.viz import plot_epi_curve

# 資料：某社區辦桌宴會後的賓客名冊（模擬諾羅病毒食因性群聚）
rng = np.random.default_rng(614)
n = 240
banquet_date = pd.Timestamp("2026-03-14")

guest_id = np.arange(1, n + 1)
table_no = rng.integers(1, 25, n)  # 24 桌
ate_oysters = rng.random(n) < 0.4  # 4 成賓客食用生蠔冷盤

# 食用生蠔冷盤者感染機率明顯較高（可疑暴露）
p_infect = np.where(ate_oysters, 0.65, 0.08)
infected = rng.random(n) < p_infect

# 諾羅病毒潛伏期短（約 12-48 小時），只有病例才有發病日
incubation_hours = rng.normal(30, 8, n).clip(10, 60)
onset_datetime = pd.DatetimeIndex(banquet_date + pd.to_timedelta(incubation_hours, unit="h"))

df6 = pd.DataFrame({
    "guest_id": guest_id,
    "table_no": table_no,
    "ate_oysters": np.where(ate_oysters, "yes", "no"),
    "infected": infected,
    "symptom_onset_date": pd.NaT,
})
df6.loc[infected, "symptom_onset_date"] = onset_datetime[infected].normalize()

# --- 暴露表：是否食用生蠔冷盤 ---
exposed = df6[df6["ate_oysters"] == "yes"]
unexposed = df6[df6["ate_oysters"] == "no"]
exposed_cases, exposed_total = int(exposed["infected"].sum()), len(exposed)
unexposed_cases, unexposed_total = int(unexposed["infected"].sum()), len(unexposed)

ar_exposed = attack_rate(exposed_cases, exposed_total)
ar_unexposed = attack_rate(unexposed_cases, unexposed_total)
rr = risk_ratio(exposed_cases, exposed_total, unexposed_cases, unexposed_total)

print("=== 暴露表：是否食用生蠔冷盤 ===")
print(f"食用生蠔：{exposed_cases}/{exposed_total}　侵襲率 {ar_exposed:.1%}")
print(f"未食用　：{unexposed_cases}/{unexposed_total}　侵襲率 {ar_unexposed:.1%}")
print(f"風險比 RR = {rr:.2f}")

# --- 流行曲線 ---
ax = plot_epi_curve(df6.dropna(subset=["symptom_onset_date"]), date_col="symptom_onset_date")
ax.set_title("辦桌宴會諾羅病毒群聚流行曲線，依發病日")
ax.set_xlabel("發病日期")
ax.set_ylabel("病例數")
plt.show()

# --- SitRep 摘要 ---
total_cases = int(df6["infected"].sum())
print("\n=== SitRep 摘要 ===")
print(f"賓客總數：{n}")
print(f"病例數：{total_cases}（整體侵襲率 {attack_rate(total_cases, n):.1%}）")
print(f"風險比 RR（食用生蠔冷盤 vs 未食用）= {rr:.2f}")

print("\n判讀：流行曲線集中在宴會後 1-2 天內出現單一高峰、隨後迅速下降，")
print("屬於典型的『點源型（point source）』群聚型態；")
print(f"食用生蠔冷盤者的感染風險是未食用者的 {rr:.1f} 倍，支持生蠔冷盤為本次群聚的可疑暴露來源。")

## 題目 7 解答

In [ ]:
import numpy as np
from epi_learning.metrics import risk_ratio
from epi_learning.tabulate import summarize_by_group
from epi_learning.viz import plot_epi_curve

# 資料：某公司部門聚餐後的員工名冊（模擬 COVID-19 職場群聚）
rng = np.random.default_rng(719)
n = 450
departments = ["業務部", "行政部", "IT部", "財務部", "客服部"]
dept_sizes = [150, 80, 70, 60, 90]
department = np.repeat(departments, dept_sizes)

# 各部門參加聚餐的比例不同（業務部聚餐參加率最高）
p_meeting = {"業務部": 0.75, "行政部": 0.30, "IT部": 0.15, "財務部": 0.20, "客服部": 0.35}
attended_meeting = np.array([rng.random() < p_meeting[d] for d in department])

# 參加聚餐者感染機率明顯較高
p_infect = np.where(attended_meeting, 0.55, 0.05)
infected = rng.random(n) < p_infect

# COVID-19 潛伏期約 2-8 天
onset_offset_days = rng.integers(2, 9, n)
meeting_date = pd.Timestamp("2026-03-01")
onset_date = meeting_date + pd.to_timedelta(onset_offset_days, unit="D")

df7 = pd.DataFrame({
    "employee_id": np.arange(1, n + 1),
    "department": department,
    "attended_meeting": np.where(attended_meeting, "yes", "no"),
    "infected": infected,
    "symptom_onset_date": pd.NaT,
})
df7.loc[infected, "symptom_onset_date"] = onset_date[infected]

# --- 各部門病例數與佔比 ---
cases7 = df7[df7["infected"]]
dept_summary = summarize_by_group(cases7, "department", "employee_id")
print("=== 各部門病例數與佔比 ===")
print(dept_summary.to_string(index=False))

# --- 各部門侵襲率 ---
dept_stats = df7.groupby("department").agg(
    n=("employee_id", "size"), infected=("infected", "sum"),
)
dept_stats["AR%"] = (dept_stats["infected"] / dept_stats["n"] * 100).round(1)
dept_stats = dept_stats.sort_values("AR%", ascending=False)
print("\n=== 各部門侵襲率（依高到低排序）===")
print(dept_stats.to_string())

# --- 流行曲線 ---
ax = plot_epi_curve(df7.dropna(subset=["symptom_onset_date"]), date_col="symptom_onset_date")
ax.set_title("公司 COVID-19 群聚流行曲線，依發病日")
ax.set_xlabel("發病日期")
ax.set_ylabel("病例數")
plt.show()

# --- 風險比：是否參加聚餐 ---
exp7 = df7[df7["attended_meeting"] == "yes"]
unexp7 = df7[df7["attended_meeting"] == "no"]
rr7 = risk_ratio(
    int(exp7["infected"].sum()), len(exp7),
    int(unexp7["infected"].sum()), len(unexp7),
)

worst_dept = dept_stats.index[0]
print(f"\n侵襲率最高部門：{worst_dept}（{dept_stats.iloc[0]['AR%']}%）")
print(f"風險比 RR（參加聚餐 vs 未參加）= {rr7:.2f}")
print("\n判讀：侵襲率最高的部門同時也是聚餐參加率最高的部門，")
print("加上參加聚餐者的感染風險顯著高於未參加者，")
print("支持部門聚餐是本次職場群聚的主要傳播熱點，應優先針對聚餐接觸者進行疫調。")

## 題目 8 解答

In [ ]:
import numpy as np
from epi_learning.metrics import attack_rate, risk_ratio
from epi_learning.viz import plot_epi_curve

# 資料：某國小麻疹群聚，模擬班級內的傳播鏈（含世代與感染源）
rng = np.random.default_rng(2026)
n_classes = 20
students_per_class = 20
n = n_classes * students_per_class  # 400 名學生

class_id = np.repeat([f"C{i+1:02d}" for i in range(n_classes)], students_per_class)
student_id = np.arange(1, n + 1)
vaccinated = rng.random(n) < 0.92  # 疫苗覆蓋率 92%（低於麻疹群體免疫閾值約 95%）

df8 = pd.DataFrame({
    "student_id": student_id,
    "class_id": class_id,
    "vaccinated": np.where(vaccinated, "yes", "no"),
})
df8["infected"] = False
df8["symptom_onset_date"] = pd.NaT
df8["infector_id"] = pd.NA
df8["generation"] = pd.NA

start_date = pd.Timestamp("2026-03-02")

# 世代 0：3 名社區感染的未接種指標病例
unvacc_idx = df8.index[df8["vaccinated"] == "no"].to_numpy()
primary_idx = rng.choice(unvacc_idx, size=3, replace=False)
for idx in primary_idx:
    df8.loc[idx, "infected"] = True
    df8.loc[idx, "symptom_onset_date"] = start_date + pd.Timedelta(days=int(rng.integers(0, 3)))
    df8.loc[idx, "generation"] = 0

# 依世代模擬班級內傳播鏈：每個病例可能傳染同班未感染的同學，
# 已接種者仍可能被感染，但機率因疫苗保護力（97%）大幅降低
vaccine_efficacy = 0.97
p_transmit_unvacc = 0.55  # 每一對同班接觸者間的傳染機率（未接種）
max_generations = 5

current_gen = 0
while current_gen < max_generations:
    infectors = df8[(df8["generation"] == current_gen) & df8["infected"]]
    if infectors.empty:
        break
    for _, case in infectors.iterrows():
        classmates = df8[
            (df8["class_id"] == case["class_id"])
            & (~df8["infected"])
            & (df8.index != case.name)
        ]
        for cm_idx, cm in classmates.iterrows():
            transmit_prob = (
                p_transmit_unvacc * (1 - vaccine_efficacy)
                if cm["vaccinated"] == "yes"
                else p_transmit_unvacc
            )
            if rng.random() < transmit_prob:
                generation_interval = max(7, rng.normal(12, 2))  # 世代間隔天數
                onset = case["symptom_onset_date"] + pd.Timedelta(days=generation_interval)
                df8.loc[cm_idx, "infected"] = True
                df8.loc[cm_idx, "symptom_onset_date"] = onset
                df8.loc[cm_idx, "infector_id"] = case["student_id"]
                df8.loc[cm_idx, "generation"] = current_gen + 1
    current_gen += 1

# --- 侵襲率與風險比：未接種 vs 已接種 ---
unvacc = df8[df8["vaccinated"] == "no"]
vacc = df8[df8["vaccinated"] == "yes"]
unvacc_cases, unvacc_total = int(unvacc["infected"].sum()), len(unvacc)
vacc_cases, vacc_total = int(vacc["infected"].sum()), len(vacc)

ar_unvacc = attack_rate(unvacc_cases, unvacc_total)
ar_vacc = attack_rate(vacc_cases, vacc_total)
rr8 = risk_ratio(unvacc_cases, unvacc_total, vacc_cases, vacc_total)

print("=== 侵襲率：疫苗接種狀態 ===")
print(f"未接種：{unvacc_cases}/{unvacc_total}　侵襲率 {ar_unvacc:.1%}")
print(f"已接種：{vacc_cases}/{vacc_total}　侵襲率 {ar_vacc:.1%}")
print(f"風險比 RR（未接種 vs 已接種）= {rr8:.1f}")

# --- 流行曲線（觀察多個世代波段） ---
ax = plot_epi_curve(df8.dropna(subset=["symptom_onset_date"]), date_col="symptom_onset_date")
ax.set_title("校園麻疹群聚流行曲線，依發病日")
ax.set_xlabel("發病日期")
ax.set_ylabel("病例數")
plt.show()

# --- 世代間隔（serial interval）估計 ---
secondary = df8.dropna(subset=["infector_id"]).copy()
secondary["infector_id"] = secondary["infector_id"].astype(int)
onset_by_id = df8.set_index("student_id")["symptom_onset_date"]
secondary["infector_onset"] = secondary["infector_id"].map(onset_by_id)
secondary["serial_interval_days"] = (
    secondary["symptom_onset_date"] - secondary["infector_onset"]
).dt.days

si_mean = secondary["serial_interval_days"].mean()
si_median = secondary["serial_interval_days"].median()

# --- SitRep 摘要 ---
total_cases8 = int(df8["infected"].sum())
print("\n=== SitRep 摘要 ===")
print(f"學生總數：{n}")
print(f"病例數：{total_cases8}（整體侵襲率 {attack_rate(total_cases8, n):.1%}）")
print(f"風險比 RR（未接種 vs 已接種）= {rr8:.1f}")
print(f"世代間隔估計：平均 {si_mean:.1f} 天，中位數 {si_median:.1f} 天（n={len(secondary)} 對傳播鏈）")

print("\n判讀：")
print("- 未接種學生的感染風險遠高於已接種學生，顯示疫苗仍是最有效的防護")
print("- 估計出的世代間隔與文獻報告的麻疹世代間隔（約 11-12 天）相近，")
print("  顯示本次群聚確實透過班級內接觸持續傳播，而非單一暴露源")
print("- 92% 的疫苗覆蓋率低於麻疹建立群體免疫所需的約 95% 閾值，")
print("  這正是校園群聚得以跨世代持續擴散的重要原因")